# FRDA-only feature selection

This notebook defines the **baseline** feature recipe. It selects progression-sensitive MRI features using FRDA outer-training participants only. Controls do not influence feature ranking.

The fold-specific recipes below are the only lists that downstream notebooks may use for unbiased out-of-fold evaluation. The final full-data list is shown separately and is intended only for final refitting, interpretation, and deployment.


## 1. Data, candidate panel, and shared folds

All models start from the same prespecified 70-feature panel. Participants, rather than annual pair rows, define the five outer folds, keeping V1->V2 and V2->V3 observations from one person together.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the repository root")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.trackfa_pairs import trackfa_pairs_to_long
from src.eval.control_aware_selection import wide_cohort_to_pair_long
from src.eval.feature_recipes import (
    build_fold_local_feature_recipe,
    build_full_data_feature_recipe,
    compare_feature_recipes,
    feature_recipe_stability,
)
from src.features.panels import a_priori_70_feature_names
from src.reporting.experiment_artifacts import (
    build_experiment_manifest,
    make_participant_folds,
    read_experiment_contract,
    read_table_artifact,
    write_experiment_contract,
    write_table_artifact,
)

repo_data_root = REPO_ROOT / "data" / "processed"
fallback_data_root = Path("/Users/robertwang/Documents/New_project/biomarkers/data/processed")
DATA_ROOT = repo_data_root if repo_data_root.is_dir() else fallback_data_root
PAIRS_PATH = DATA_ROOT / "trackfa_pairs_drop3poms.csv"
WIDE_PATH = DATA_ROOT / "trackfa_merged_wide.csv"
for required in (PAIRS_PATH, WIDE_PATH):
    if not required.is_file():
        raise FileNotFoundError(f"Required input not found: {required}")

RUN_ID = "trackfa_70_feature_comparison_v1"
RUN_DIR = REPO_ROOT / "results" / "experiments" / RUN_ID
SELECTION_DIR = RUN_DIR / "selections"
SELECTION_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
OUTER_FOLDS = 5
INNER_FOLDS = 5
FEATURE_COUNT = 16
CONTROL_PENALTY = 0.5
feature_names = a_priori_70_feature_names()
assert len(feature_names) == 70 and len(set(feature_names)) == 70

pairs = pd.read_csv(PAIRS_PATH, low_memory=False)
frda_long = trackfa_pairs_to_long(pairs)
missing_frda = [feature for feature in feature_names if feature not in frda_long.columns]
if missing_frda:
    raise KeyError(f"FRDA data are missing panel features: {missing_frda}")


control_ids = pd.read_csv(WIDE_PATH, usecols=["ID", "study_group"], low_memory=False)
control_ids = control_ids.loc[pd.to_numeric(control_ids["study_group"], errors="coerce").eq(1), "ID"]
control_ids = control_ids.astype(str).str.replace(r"^TRACKFA_", "", regex=True)

folds = make_participant_folds(
    {
        "FRDA": sorted(frda_long["subject"].astype(str).unique()),
        "Control": sorted(control_ids.unique()),
    },
    n_splits=OUTER_FOLDS,
    seed=SEED,
)
manifest = build_experiment_manifest(
    run_id=RUN_ID,
    data_path=PAIRS_PATH,
    feature_panel=feature_names,
    folds=folds,
    seed=SEED,
    outer_splits=OUTER_FOLDS,
    inner_splits=INNER_FOLDS,
    source_notebooks=["feature_selection_pipeline.ipynb", "control_aware_feature_selection.ipynb"],
    strategy_definitions={
        "frda_only": "Top 16 interval-sensitive marginal FRDA progression features from each outer-training fold",
        "control_aware": "Top 16 interval-consistent FRDA features after same-direction control penalty lambda=0.5",
    },
    clinical_heads_enabled=False,
)
write_experiment_contract(RUN_DIR, manifest=manifest, folds=folds)

panel_audit = pd.DataFrame({"Panel order": np.arange(1, 71), "Feature": feature_names})
fold_counts = folds.groupby(["cohort", "outer_fold"], as_index=False).agg(Participants=("participant_id", "nunique"))
print(f"Experiment: {RUN_ID}")
print(f"Candidate panel: {len(feature_names)} MRI features")
display(fold_counts.rename(columns={"cohort": "Cohort", "outer_fold": "Outer fold"}))
display(panel_audit)


Experiment: trackfa_70_feature_comparison_v1
Candidate panel: 70 MRI features


,Cohort,Outer fold,Participants
0,Control,1,19
1,Control,2,19
2,Control,3,19
3,Control,4,19
4,Control,5,19
5,FRDA,1,24
6,FRDA,2,24
7,FRDA,3,23
8,FRDA,4,23
9,FRDA,5,23


,Panel order,Feature
0,1,Cerebellum_WM_CerebNet
1,2,Cerebellum_Cortex_CerebNet
2,3,SCP
3,4,Medulla
4,5,Pons
...,...,...
65,66,RD_gCC
66,67,RD_mLEM
67,68,RD_sCC
68,69,sFA_c3c5


## 2. Fold-local FRDA ranking

Within each outer fold, features are ranked only from the remaining FRDA participants. The ranking prioritises the absolute mean of the V1->V2 and V2->V3 paired Cohen's $d_z$, with the interval gap used as a deterministic tie-breaker.


In [2]:
baseline_recipe = build_fold_local_feature_recipe(
    frda_long,
    feature_names,
    folds,
    strategy="frda_only",
    k=FEATURE_COUNT,
)
write_table_artifact(
    SELECTION_DIR / "frda_only_features_by_fold.csv",
    baseline_recipe,
    schema="feature_recipe",
    manifest=manifest,
    metadata={"scope": "outer-training folds", "k": FEATURE_COUNT, "uses_controls": False},
    panel=feature_names,
)

selected_by_fold = (
    baseline_recipe.loc[baseline_recipe["selected"]]
    .groupby("outer_fold", as_index=False)
    .agg(
        Selected_features=("feature", "size"),
        Training_FRDA_participants=("train_frda_participants", "first"),
        Training_FRDA_pairs=("train_frda_pairs", "first"),
        Features=("feature", lambda values: " | ".join(values)),
    )
    .rename(columns={"outer_fold": "Outer fold"})
)
display(selected_by_fold)


,Outer fold,Selected_features,Training_FRDA_participants,Training_FRDA_pairs,Features
0,1,16,93,165,Cerebellum_Cortex_CerebNet | Pons | Cerebellum...
1,2,16,93,163,Cerebellum_Cortex_CerebNet | Pons | TotalBrain...
2,3,16,94,167,Cerebellum_Cortex_CerebNet | Pons | Thalamus |...
3,4,16,94,166,Cerebellum_Cortex_CerebNet | Pons | Cerebellum...
4,5,16,94,167,Cerebellum_Cortex_CerebNet | Pons | Cerebellum...


## 3. Marginal effects and selection stability

The effect sizes are marginal single-feature changes, not multivariable model coefficients. Values are calculated from each outer-training fold and therefore vary slightly across folds.


In [3]:
baseline_frequency, baseline_jaccard = feature_recipe_stability(baseline_recipe)
rank_summary = (
    baseline_recipe.groupby("feature", as_index=False)
    .agg(
        Mean_rank=("rank", "mean"),
        Mean_FRDA_pooled_d=("frda_pooled_d_z", "mean"),
        Mean_FRDA_V1_V2_d=("frda_v1_v2_d_z", "mean"),
        Mean_FRDA_V2_V3_d=("frda_v2_v3_d_z", "mean"),
    )
    .merge(baseline_frequency[["feature", "selection_frequency"]], on="feature", how="left")
    .fillna({"selection_frequency": 0.0})
    .sort_values(["selection_frequency", "Mean_rank"], ascending=[False, True], kind="mergesort")
)
rank_summary.insert(0, "Rank", np.arange(1, len(rank_summary) + 1))
rank_summary = rank_summary.rename(columns={"feature": "Feature", "selection_frequency": "Selection frequency"})
display(rank_summary.head(25).round(3))

jaccard_summary = pd.DataFrame([{
    "Strategy": "FRDA-only",
    "Outer folds": OUTER_FOLDS,
    "Selected per fold": FEATURE_COUNT,
    "Mean Jaccard": baseline_jaccard["jaccard"].mean(),
    "Minimum Jaccard": baseline_jaccard["jaccard"].min(),
    "Maximum Jaccard": baseline_jaccard["jaccard"].max(),
}])
display(jaccard_summary.round(3))
baseline_frequency.to_csv(SELECTION_DIR / "frda_only_selection_frequency.csv", index=False)
baseline_jaccard.to_csv(SELECTION_DIR / "frda_only_selection_jaccard.csv", index=False)


,Rank,Feature,Mean_rank,Mean_FRDA_pooled_d,Mean_FRDA_V1_V2_d,Mean_FRDA_V2_V3_d,Selection frequency
1,1,Cerebellum_Cortex_CerebNet,1.0,-0.621,-0.881,-0.403,1.0
34,2,Pons,2.0,-0.552,-0.778,-0.359,1.0
2,3,Cerebellum_WM_CerebNet,3.6,-0.426,-0.512,-0.348,1.0
65,4,TotalBrainGMVol_nocereb,4.2,-0.413,-0.405,-0.425,1.0
30,5,Lateral_Ventricle,4.8,0.386,0.537,0.288,1.0
32,6,Midbrain,6.8,-0.353,-0.389,-0.325,1.0
64,7,Thalamus,6.8,-0.369,-0.475,-0.249,1.0
35,8,Putamen,7.8,-0.339,-0.423,-0.240,1.0
31,9,Medulla,8.2,-0.331,-0.546,-0.129,1.0
18,10,FA_PTR,11.0,-0.254,-0.208,-0.305,1.0


,Strategy,Outer folds,Selected per fold,Mean Jaccard,Minimum Jaccard,Maximum Jaccard
0,FRDA-only,5,16,0.788,0.778,0.882


## 4. Full-data recipe for final refitting only

This list uses all available FRDA participants. It must not be used to estimate cross-validated performance because it has seen the complete FRDA dataset.


In [4]:
baseline_full = build_full_data_feature_recipe(
    frda_long,
    feature_names,
    strategy="frda_only",
    k=FEATURE_COUNT,
)
write_table_artifact(
    SELECTION_DIR / "frda_only_full_data_features.csv",
    baseline_full,
    schema="feature_recipe",
    manifest=manifest,
    metadata={"scope": "full-data deployment only", "k": FEATURE_COUNT, "uses_controls": False},
    panel=feature_names,
)
final_baseline_display = baseline_full.loc[baseline_full["selected"], [
    "rank", "feature", "selection_score", "frda_pooled_d_z", "frda_v1_v2_d_z", "frda_v2_v3_d_z"
]].rename(columns={
    "rank": "Rank", "feature": "Feature", "selection_score": "Selection score",
    "frda_pooled_d_z": "FRDA pooled d", "frda_v1_v2_d_z": "FRDA V1->V2 d",
    "frda_v2_v3_d_z": "FRDA V2->V3 d",
})
display(final_baseline_display.round(3))
print(f"Validated fold-specific recipe: {SELECTION_DIR / 'frda_only_features_by_fold.csv'}")
print(f"Deployment-only recipe: {SELECTION_DIR / 'frda_only_full_data_features.csv'}")


,Rank,Feature,Selection score,FRDA pooled d,FRDA V1->V2 d,FRDA V2->V3 d
0,1,Cerebellum_Cortex_CerebNet,0.641,-0.620,-0.878,-0.403
1,2,Pons,0.568,-0.552,-0.777,-0.359
2,3,Cerebellum_WM_CerebNet,0.426,-0.425,-0.510,-0.342
3,4,TotalBrainGMVol_nocereb,0.412,-0.413,-0.402,-0.423
4,5,Lateral_Ventricle,0.411,0.385,0.536,0.286
5,6,Thalamus,0.359,-0.367,-0.469,-0.248
6,7,Midbrain,0.356,-0.352,-0.388,-0.324
7,8,Medulla,0.336,-0.331,-0.545,-0.127
8,9,Putamen,0.331,-0.339,-0.422,-0.240
9,10,Caudate,0.260,-0.256,-0.223,-0.296


Validated fold-specific recipe: /Users/robertwang/Documents/New_project/biomarkers/results/experiments/trackfa_70_feature_comparison_v1/selections/frda_only_features_by_fold.csv
Deployment-only recipe: /Users/robertwang/Documents/New_project/biomarkers/results/experiments/trackfa_70_feature_comparison_v1/selections/frda_only_full_data_features.csv


## Interpretation

This notebook establishes the non-control-focused baseline. It does not decide which model is best and does not report final model coefficients or clinical benchmarks. Model performance is evaluated later using held-out FRDA and control participants.
